# 1. Selección de Datos y Planteamiento de Hipótesis

## 1.1 Justificación de la Fuente de Datos
Para este proyecto, hemos decidido trabajar con el dataset de **Precios Históricos del S&P 500**. 

**Se eligieron 11 empresas, pero ¿Porque se seleccionaron especificamente esas 11?**
No elegimos empresas al azar, se selecciono un portafolio que mezcla dos mundos opuestos:
1.  **Los Gigantes Tecnológicos:** Apple, Microsoft, Nvidia, Google, Amazon, Meta, Netflix y Tesla. Son empresas que crecen mucho pero son muy sensibles a las crisis.
2.  **El Consumo Estable:** Coca-Cola, Pepsi y McDonald's. Son empresas que la gente sigue usando incluso si hay crisis, por lo que sirven como "punto de control" para comparar qué tan inestables son los movimientos de las empresas tecnológicas.

---

## 1.2 Hipótesis y Expectativas del Análisis
*Antes de analizar los resultados finales, planteamos lo que nuestra intuición (y las noticias recientes) nos dice que deberíamos encontrar:*

### Hipótesis 1: El Shock Global de la Pandemia (Marzo 2020)
**Contexto:** En marzo de 2020, el mundo se detuvo por el COVID-19. 
**Hipótesis:** Esperamos ver una "caída en picada" casi vertical en todas las líneas del gráfico. Sin embargo, nuestra hipótesis es que las empresas tecnológicas de Streaming (como Amazon o Netflix) se recuperarán mucho más rápido que las demás, ya que el confinamiento obligó a todo el mundo a usar servicios digitales, acelerando su crecimiento.



### Hipótesis 2: El Auge de la Inteligencia Artificial (2023-2024)
**Contexto:** A finales de 2022 y principios de 2023, la IA (como ChatGPT) cambió las reglas del juego.
**Hipótesis:** Predecimos que empresas de hardware y software como **Nvidia** y **Microsoft** mostrarán una pendiente de subida mucho más empinada que el resto del mercado. Creemos que este será el evento más visible en la segunda mitad de nuestro gráfico, marcando una diferencia gigante con empresas tradicionales.



### Hipótesis 3: La Diferencia de Volatilidad
**Contexto:** No todas las acciones se mueven igual. 
**Hipótesis:** Esperamos que las líneas de **Coca-Cola (KO)** y **Pepsi (PEP)** sean mucho más suaves y persistentes. Por el contrario, empresas como **Tesla (TSLA)** o **Nvidia (NVDA)** deberían mostrar picos y valles muy pronunciados. Esto nos servirá para demostrar que el riesgo de inversión es mucho mayor en el sector tecnológico que en el de consumo básico.

### Hipótesis 4: La Caída por Inflación (2022)
**Contexto:** Durante el 2022, hubo mucha incertidumbre económica y subida de precios a nivel mundial.
**Hipótesis:** Esperamos ver un periodo de decadencia donde la mayoría de los activos perdieron valor. Nuestra intuición es que este periodo servirá para limpiar el exceso de optimismo de la post-pandemia antes del nuevo salto de la IA.

In [2]:
#Cargamos todas las librerias necesarias para el proyecto.
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import requests #Libreria necesaria para adquirir datos desde una API 
import io #Libreria necesaria para descomprimir archivo .zip de la base de datos sin necesidad de descargarlo en disco.
from collections import Counter #Libreria para contar.
import mplfinance as mpf #Libreria especializada en hacer graficos de velas con volumen.
from statsmodels.graphics.tsaplots import plot_acf #Libreria para medir la autocorrelacion de precios dependiendo del tiempo.
from statsmodels.tsa.seasonal import seasonal_decompose #Libreria que grafica la Tendencia, Estacionalidad y Residuo de las acciones. 

In [4]:
#Se comienza la obtencion de datos desde la API

#URL de la API que utilizaremos
URL = "https://www.kaggle.com/api/v1/datasets/download/yash16jr/s-and-p500-daily-update-dataset"

#Se le piden los datos a la API de Kaggle
response = requests.get(URL)

#Nos imprime el codigo de respuesta, si el codigo es [200] se adquirio de manera correcta la base de datos desde la API.
print(response)

#Se verifica el tamaño del archivo para ver si se adquirio completa la base de datos, debiendo pesar aprox 71mb
print(f"El tamaño del archivo es de {len(response.content)*1e-6} mb") 

<Response [200]>
El tamaño del archivo es de 70.579545 mb


In [5]:
#Creamos el objeto con los datos
datos = io.BytesIO(response.content)

#Cargamos directamente a un DataFrame (Pandas se encarga de descomprimir el .zip)
df = pd.read_csv(datos, compression="zip", low_memory=False)

print("Datos Cargados con Exito!")

Datos Cargados con Exito!


## 2. Analisis de Datos Crudos
 
*Se procede a hacer un analisis inicial de los datos para ver como estan distribuidos los datos y que tipo de datos se tienen en la muestra.*

In [8]:
#Vista inicial de los primeros datos utilizando .head()
print("Los primeros datos son:")
df.head()

Los primeros datos son:


,Price,Close,Close.1,Close.2,Close.3,Close.4,Close.5,Close.6,Close.7,Close.8,Close.9,Close.10,Close.11,Close.12,Close.13,Close.14,Close.15,Close.16,Close.17,Close.18,Close.19,Close.20,Close.21,Close.22,Close.23,Close.24,Close.25,Close.26,Close.27,Close.28,Close.29,Close.30,Close.31,Close.32,Close.33,Close.34,Close.35,Close.36,Close.37,Close.38,...,Volume.463,Volume.464,Volume.465,Volume.466,Volume.467,Volume.468,Volume.469,Volume.470,Volume.471,Volume.472,Volume.473,Volume.474,Volume.475,Volume.476,Volume.477,Volume.478,Volume.479,Volume.480,Volume.481,Volume.482,Volume.483,Volume.484,Volume.485,Volume.486,Volume.487,Volume.488,Volume.489,Volume.490,Volume.491,Volume.492,Volume.493,Volume.494,Volume.495,Volume.496,Volume.497,Volume.498,Volume.499,Volume.500,Volume.501,Volume.502
0,Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,ADSK,AEE,AEP,AES,AFL,AIG,AIZ,AJG,AKAM,ALB,ALGN,ALL,ALLE,AMAT,AMCR,AMD,AME,AMGN,AMP,AMT,AMZN,ANET,AON,AOS,APA,APD,APH,APO,...,URI,USB,V,VICI,VLO,VLTO,VMC,VRSK,VRSN,VRT,VRTX,VST,VTR,VTRS,VZ,WAB,WAT,WBD,WDAY,WDC,WEC,WELL,WFC,WM,WMB,WMT,WRB,WSM,WST,WTW,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
1,Date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-04,19.856182098388672,6.412384033203125,NaN,NaN,18.321969985961914,7.60190486907959,31.492164611816406,37.09000015258789,21.829465866088867,20.289302825927734,25.521587371826172,25.670000076293945,15.399174690246582,18.71431541442871,8.72465705871582,16.052507400512695,18.67699432373047,21.96930694580078,14.929019927978516,25.920000076293945,30.16242790222168,18.5,21.10109519958496,NaN,11.013404846191406,NaN,9.699999809265137,15.443207740783691,39.041160583496094,28.407115936279297,31.776994705200195,6.695000171661377,NaN,32.12552261352539,5.788906097412109,77.2918701171875,51.43324661254883,5.040172100067139,NaN,...,1692500,12891700,20180000,NaN,15454500,NaN,1128000,390000,2652000,NaN,1744900,NaN,1048125,3611900,16176648,510600,954400,2844108,NaN,4125379,1541800,1426600,39335700,2058800,7020240,62259300,6416888,3979400,232800,348017,1832400,4741400,2670400,27809100,NaN,NaN,2962274,805872,168800,NaN
3,2010-01-05,19.640501022338867,6.423468589782715,NaN,NaN,18.1739444732666,7.5765485763549805,31.68680763244629,37.70000076293945,21.795007705688477,20.398900985717773,25.38453483581543,25.280000686645508,15.338149070739746,18.500070571899414,8.635306358337402,16.518190383911133,18.327064514160156,22.624887466430664,14.848797798156738,26.690000534057617,30.10700035095215,18.010000228881836,21.448041915893555,NaN,10.92868423461914,NaN,9.710000038146973,15.375186920166016,38.70296096801758,29.0695743560791,32.30345916748047,6.734499931335449,NaN,31.92229652404785,5.714937686920166,78.20442199707031,51.00647735595703,4.934847831726074,NaN,...,1459200,14503500,25833600,NaN,17087295,NaN,830300,430000,4628700,NaN,2456500,NaN,1265036,7942400,23722957,587600,1545000,3193041,NaN,9325695,1626000,1173300,55416000,2961700,7415481,46945200,3860325,4343200,245800,339523,1724500,5644300,4321400,30174700,NaN,NaN,3298757,1769643,168800,NaN
4,2010-01-06,19.570714950561523,6.321296215057373,NaN,NaN,18.274869918823242,7.543794631958008,32.02366638183594,37.619998931884766,21.753658294677734,20.347328186035156,25.32494354248047,25.34000015258789,15.23275089263916,18.687538146972656,8.545950889587402,16.66329574584961,18.208351135253906,22.559337615966797,14.855484962463379,26.469999313354492,30.16242790222168,17.479999542236328,21.45498275756836,NaN,10.905580520629883,NaN,9.569999694824219,15.391200065612793,38.4121208190918,29.47559928894043,32.47161865234375,6.612500190734863,NaN,31.913833618164062,5.717532157897949,79.43094635009766,50.58589172363281,4.953497409820557,NaN,...,1072900,12293600,16254000,NaN

In [9]:
#Vista inicial de los ultimos datos utilizando .tail()
print("Los ultimos datos son:")
df.tail()

Los ultimos datos son:


,Price,Close,Close.1,Close.2,Close.3,Close.4,Close.5,Close.6,Close.7,Close.8,Close.9,Close.10,Close.11,Close.12,Close.13,Close.14,Close.15,Close.16,Close.17,Close.18,Close.19,Close.20,Close.21,Close.22,Close.23,Close.24,Close.25,Close.26,Close.27,Close.28,Close.29,Close.30,Close.31,Close.32,Close.33,Close.34,Close.35,Close.36,Close.37,Close.38,...,Volume.463,Volume.464,Volume.465,Volume.466,Volume.467,Volume.468,Volume.469,Volume.470,Volume.471,Volume.472,Volume.473,Volume.474,Volume.475,Volume.476,Volume.477,Volume.478,Volume.479,Volume.480,Volume.481,Volume.482,Volume.483,Volume.484,Volume.485,Volume.486,Volume.487,Volume.488,Volume.489,Volume.490,Volume.491,Volume.492,Volume.493,Volume.494,Volume.495,Volume.496,Volume.497,Volume.498,Volume.499,Volume.500,Volume.501,Volume.502
4080,2026-03-23,112.0199966430664,251.49000549316406,204.92999267578125,132.58999633789062,104.8499984741211,93.66000366210938,200.02000427246094,247.63999938964844,312.19000244140625,67.98999786376953,209.7100067138672,247.44000244140625,106.9000015258789,127.91999816894531,14.079999923706055,106.6500015258789,75.08999633789062,217.42999267578125,216.74000549316406,114.43000030517578,167.55999755859375,180.86000061035156,207.75999450683594,143.6300048828125,361.7900085449219,39.36000061035156,202.67999267578125,212.80999755859375,349.7699890136719,442.9100036621094,176.5,210.13999938964844,135.8800048828125,325.9700012207031,65.06999969482422,39.029998779296875,278.6600036621094,130.6699981689453,110.44999694824219,...,570700,12235400,7824500,11009700.0,3904200,2987000.0,1312000,1559600,546900,8888100.0,1152000,5338500.0,2925100,10566700,26390300,761200,963000,29999900,4493900.0,8990500,1644400,3598100,15320800,2043600,6519400,22181500,2498200,1597600,845300,461000,5768200,1605200,5888500,25021300,2668200.0,6577000.0,1771600,1782600,602400,4617900.0
4081,2026-03-24,114.19999694824219,251.63999938964844,205.1999969482422,130.0,104.05999755859375,93.5999984741211,193.5399932861328,238.8699951171875,321.8299865722656,71.44000244140625,204.88999938964844,239.38999938964844,107.69000244140625,128.8000030517578,14.130000114440918,106.19999694824219,74.33999633789062,217.52000427246094,216.27999877929688,114.5,177.05999755859375,179.33999633789062,207.30999755859375,145.72999572753906,373.989990234375,39.130001068115234,205.3699951171875,216.97999572753906,348.42999267578125,448.1700134277344,170.36000061035156,207.24000549316406,130.8000030517578,327.0299987792969,65.44999694824219,40.79999923706055,286.25,127.95999908447266,111.25,...,495300,13434600,5375900,9993000.0,3678300,1494700.0,1699300,1496100,626800,6843000.0,1224200,3995500.0,2689800,7651200,20264600,760400,671000,29087600,5236600.0,7194700,1824500,2598700,19134300,1775800,4725800,17269600,1800200,1158000,997100,500400,4609200,1079500,4313000,26131800,2081200.0,4684000.0,1319500,1572200,425000,4561800.0
4082,2026-03-25,112.9800033569336,252.6199951171875,207.17999267578125,131.80999755859375,104.83000183105469,93.31999969482422,192.2899932861328,237.25,322.0299987792969,71.66000366210938,202.11000061035156,235.4199981689453,107.80000305175781,128.3000030517578,14.0600004196167,107.20999908447266,75.47000122070312,217.02000427246094,213.55999755859375,119.1500015258789,181.38999938964844,180.27000427246094,204.7100067138672,147.47999572753906,369.3399963378906,40.54999923706055,220.27000427246094,215.3300018310547,353.92999267578125,451.8900146484375,167.27000427246094,211.7100067138672,135.00999450683594,321.45001220703125,66.9000015258789,41.31999969482422,290.0899963378906,128.72999572753906,109.80000305175781,...,482100,8957400,6280900,7564900.0,4480900,2027300.0,1121300,2727200,1297600,5929000.0,1200000,3512900.0,1937200,5750700,21920200,834100,1117000,26247200,5293000.0,7642100,1424600,3036100,17213400,1919500,5035800,16597300,2661700,946900,811100,542300,3604700,1387800,2941800,17192300,1542400.0,4585500.0,1426900,1738100,456800,3078900.0
4083,2026-03-26,113.4800033569336,252.88999938964844

In [27]:
#Analisis inicial
print("1. Dimensiones del DataSet")
print(f"La muestra tiene {df.shape[0]} Filas y {df.shape[1]} Columnas")

print("\n2. Tipos de Variables")
print("Los datos son del tipo:")
#Nota: Hacemos .iloc ya que la primera fila es strings y la segunda pareciera que son NaN 
valor_fila3 = df.iloc[2].apply(type) 
print(valor_fila3)


print("\n3. Rango Temporal")
#Nota: Utilizaremos .iloc porque las fechas se encuentran en la columna 0 y desde la fila 2 en adelante.
fechas = pd.to_datetime(df.iloc[2:,0])
print(f"Los datos comienzan en la fecha {fechas.min()} y terminan en {fechas.max()}")
print(f"Por lo que el tiempo total de la muestra es de {fechas.max()-fechas.min()}")

print("\n4. Conteo de Valores Nulos (Total en la Muestra)")
print(f"En la muestra existen {df.isnull().sum().sum()} datos nulos")

1. Dimensiones del DataSet
La muestra tiene 4085 Filas y 2516 Columnas

2. Tipos de Variables
Los datos son del tipo:
Price           <class 'str'>
Close           <class 'str'>
Close.1         <class 'str'>
Close.2       <class 'float'>
Close.3       <class 'float'>
                   ...       
Volume.498    <class 'float'>
Volume.499      <class 'str'>
Volume.500      <class 'str'>
Volume.501      <class 'str'>
Volume.502    <class 'float'>
Name: 2, Length: 2516, dtype: object

3. Rango Temporal
Los datos comienzan en la fecha 2010-01-04 00:00:00 y terminan en 2026-03-27 00:00:00
Por lo que el tiempo total de la muestra es de 5926 days 00:00:00

4. Conteo de Valores Nulos (Total en la Muestra)
En la muestra existen 603735 datos nulos


### 2.1 Interpretación de Hallazgos Preliminares

Tras ejecutar el diagnóstico inicial sobre los datos crudos de la muestra, se identifican los siguientes puntos críticos que definen el objetivo para la fase de limpieza:

* **Discrepancia en la Densidad Temporal:** Se observa un rango de **5,926 días de calendario**, pero solo **4,083 filas donde se registran dias** efectivos. No se puede asumir que esta diferencia corresponde exclusivamente a feriados, existe la posibilidad de tener **vacíos de información o fallos en la captura de datos** por parte de la API. Esta incertidumbre será un factor a considerar al evaluar la continuidad de las series de tiempo.

* **Naturaleza de las Variables (Tipos de Datos):** Al revisar el resumen de la muestra, nos dimos cuenta de que las columnas de precios aparecen marcadas como **texto (`str`)** en lugar de números. Esto sucede porque en las primeras filas del archivo se colaron palabras como "Ticker", "A" o "Date", que vienen de la configuración original de la API. Como Python encuentra estas palabras al principio, asume que toda la columna es de texto. Esto hace que el dataset sea **matemáticamente inutilizable** por el momento, ya que no se pueden realizar cálculos de rendimiento, promedios o gráficos de volatilidad con "palabras". Para avanzar, será obligatorio limpiar esas filas de títulos y transformar los datos restantes en números reales para que el programa pueda procesarlos correctamente.

* **Integridad y Dispersión del Dataset:** La presencia de **603,735 valores nulos** sugiere que estamos ante una "matriz dispersa". Dado que manejamos 2,516 columnas, es altamente probable que muchos activos no posean registros para la totalidad del periodo (2010-2026). Esto justifica la necesidad de un proceso posterior de filtrado para conservar solo aquellos activos con datos íntegros.

* **Conclusión del Diagnóstico:** La muestra posee un alcance temporal que cubre hitos históricos clave, pero su calidad actual es baja debido a la inconsistencia de tipos y la alta tasa de nulos. Estos resultados validan la necesidad de proceder con una fase de **limpieza de datos** para transformar estos datos crudos en información analizable.

## 2.2 Limpieza de Datos y Eleccion de Empresas
*Ya con los datos crudos analizados se encontraron diversos problemas, debido a esto se procedera a realizar la limpieza de los datos y la eleccion de las empresas a las cuales se les realizara el analisis final.*